In [ ]:
FULL_BASE_DIR = "/kaggle/input/notebooks/ahmedamrabolfadl/opus-labeler/final_model"
LORA_BASE_DIR = "/kaggle/input/notebooks/ahmedamrabolfadl/lora-flan-t5-base-labeler/final_model"
LORA_LARGE_DIR = "/kaggle/input/notebooks/ahmedamrabolfadl/lora-flan-t5-large-labeler/final_model"


# Tokenization lengths
MAX_INPUT_LEN  = 128
MAX_TARGET_LEN = 32

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece sentence-transformers evaluate

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    get_cosine_schedule_with_warmup,
)
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
import warnings
warnings.filterwarnings("ignore")

# ── Reproducibility ──
SEED = 42
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ── Device ──
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpus = torch.cuda.device_count()
print(f"Device: {device} | GPUs available: {n_gpus}")
if torch.cuda.is_available():
    for i in range(n_gpus):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
PROMPT_TEMPLATE = (
    "Generate a concise topic label for the following.\n"
    "Research field: {field}\n"
    "Keywords: {words}\n"
    "Topic label:"
)

In [ ]:
# ── Reload model for inference demo ──
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

full_base_tokenizer = AutoTokenizer.from_pretrained(FULL_BASE_DIR)
full_base_model = AutoModelForSeq2SeqLM.from_pretrained(FULL_BASE_DIR).to(device)
full_base_model.eval()

lora_base_tokenizer = AutoTokenizer.from_pretrained(LORA_BASE_DIR)
lora_base_model = AutoModelForSeq2SeqLM.from_pretrained(LORA_BASE_DIR).to(device)
lora_base_model.eval()

lora_large_tokenizer = AutoTokenizer.from_pretrained(LORA_LARGE_DIR)
lora_large_model = AutoModelForSeq2SeqLM.from_pretrained(LORA_LARGE_DIR).to(device)
lora_large_model.eval()


def predict_topic_label(research_field: str, keywords: str, inf_tokenizer, inf_model) -> str:
    """Generate a topic label given a research field and comma-separated keywords."""
    prompt = PROMPT_TEMPLATE.format(field=research_field, words=keywords)
    inputs = inf_tokenizer(prompt, max_length=MAX_INPUT_LEN, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        output = inf_model.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )
    return inf_tokenizer.decode(output[0], skip_special_tokens=True).strip()

# ── Example predictions ──
examples = [
    ("Machine Learning", "neural, network, deep, learning, convolutional"),
    ("Economics", "inflation, monetary, policy, central, bank"),
    ("Biology", "gene, expression, regulation, transcription, protein"),
    ("Computer Science", "graph, algorithm, optimization, complexity, polynomial"),
]

print(f"{'Research Field':<25} {'Keywords':<55} {'Predicted Label'}")
print("─" * 120)
for field, keywords in examples:
    full_base_label = predict_topic_label(field, keywords, full_base_tokenizer, full_base_model)
    lora_base_label = predict_topic_label(field, keywords, lora_base_tokenizer, lora_base_model)
    lora_large_label = predict_topic_label(field, keywords, lora_large_tokenizer, lora_large_model)

    print(f"{field:<25} {keywords:<55} {full_base_label} {lora_base_label} {lora_large_label}")

In [ ]:
import pandas as pd
df = pd.read_csv('/kaggle/input/datasets/ahmedamrabolfadl/scientific-topic-labeling-dataset/testing_pairs.csv')
df = df.dropna()
print(df.shape)
df.head()

In [ ]:
df['full_base_label'] = df.apply(
    lambda row: predict_topic_label(row['research_field'], row['topic_words'], full_base_tokenizer, full_base_model), 
    axis=1
)
df['lora_base_label'] = df.apply(
    lambda row: predict_topic_label(row['research_field'], row['topic_words'], lora_base_tokenizer, lora_base_model), 
    axis=1
)
df['lora_large_label'] = df.apply(
    lambda row: predict_topic_label(row['research_field'], row['topic_words'], lora_large_tokenizer, lora_large_model), 
    axis=1
)
df.head()

In [ ]:
df.to_csv('flan_labels.csv')